# Общая оценка моделей

Единый ноутбук для тестирования всех вариантов проекта на фиксированных 100 примерах GQA-ru и 100 примерах MMBench-ru. Он считает Exact Match для GQA-ru, Accuracy для MMBench-ru и BERTScore для семантической близости открытых ответов GQA-ru.

Если файл предсказаний модели уже существует, он используется повторно. Если файла нет, но адаптер обучен, ноутбук сам загружает модель и проводит генерацию. Необученные варианты пропускаются.

In [1]:
import gc
import os
import re
import time
from pathlib import Path

import pandas as pd
import torch
from bert_score import score
from datasets import load_dataset
from peft import PeftModel
from transformers import (
    AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig,
    Qwen2_5_VLForConditionalGeneration,
)

os.environ['PYTHONIOENCODING'] = 'utf-8'
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RESULTS_DIR = ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
BASELINE_PATH = RESULTS_DIR / 'baseline_predictions.csv'
BERTSCORE_MODEL = 'xlm-roberta-large'
BERTSCORE_DEVICE = 'cpu'
assert BASELINE_PATH.exists(), 'Сначала выполните baseline-оценку из ноутбука 02.'

MODEL_REGISTRY = {
    'F1-fast': {
        'kind': 'qwen', 'base_id': 'Qwen/Qwen2.5-VL-3B-Instruct',
        'adapter': ROOT / 'models' / 'f1_fast_qlora_adapter',
        'predictions': RESULTS_DIR / 'f1_fast_predictions.csv',
    },
    'F2': {
        'kind': 'qwen', 'base_id': 'Qwen/Qwen2.5-VL-3B-Instruct',
        'adapter': ROOT / 'models' / 'f2_qwen_r16_adapter',
        'predictions': RESULTS_DIR / 'f2_predictions.csv',
    },
    'F3-10k': {
        'kind': 'qwen', 'base_id': 'Qwen/Qwen2.5-VL-3B-Instruct',
        'adapter': ROOT / 'models' / 'f3_10k_qwen_r16_adapter',
        'predictions': RESULTS_DIR / 'f3_10k_predictions.csv',
    },
    'A1-InternVL3': {
        'kind': 'internvl', 'base_id': 'OpenGVLab/InternVL3-2B-hf',
        'adapter': ROOT / 'models' / 'a1_internvl3_2b_qlora_adapter',
        'predictions': RESULTS_DIR / 'a1_internvl3_predictions.csv',
    },
}
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'не используется')

GPU: NVIDIA GeForce RTX 4060


## 1. Фиксированный тестовый набор

Идентификаторы берутся из baseline-предсказаний. Поэтому все версии получают одни и те же изображения и вопросы.

In [2]:
baseline = pd.read_csv(BASELINE_PATH)
gqa_baseline = baseline.query("benchmark == 'GQA-ru'").copy()
mmbench_baseline = baseline.query("benchmark == 'MMBench-ru'").copy()

gqa_questions = load_dataset('deepvk/GQA-ru', 'testdev_balanced_instructions', split='testdev')
gqa_images = load_dataset('deepvk/GQA-ru', 'testdev_balanced_images', split='testdev')
image_by_id = {str(row['id']): row['image'] for row in gqa_images}
gqa_by_id = {str(row['id']): row for row in gqa_questions}
gqa_eval = []
for sample_id in gqa_baseline['sample_id'].astype(str):
    row = gqa_by_id[sample_id]
    gqa_eval.append({
        'id': sample_id, 'image': image_by_id[str(row['imageId'])],
        'question': row['question'], 'answer': row['answer'],
    })

mmbench = load_dataset('deepvk/MMBench-ru', split='dev')
mmbench_by_id = {str(row['index']): row for row in mmbench}
mmbench_eval = [mmbench_by_id[str(sample_id)] for sample_id in mmbench_baseline['sample_id']]
assert len(gqa_eval) == 100 and len(mmbench_eval) == 100
print('GQA-ru: 100 | MMBench-ru: 100')

GQA-ru: 100 | MMBench-ru: 100


## 2. Генерация предсказаний для новых адаптеров

In [3]:
def normalize_gqa(text):
    return re.sub(r'[^а-яёa-z0-9-]', '', str(text).lower())

def extract_option(text):
    match = re.search(r'(?<![A-ZА-Я])[ABCD](?![A-ZА-Я])', str(text).upper())
    return match.group(0) if match else ''

def load_variant(spec):
    quantization = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
    )
    processor = AutoProcessor.from_pretrained(spec['adapter'])
    if spec['kind'] == 'internvl':
        processor.image_processor.max_patches = 1
    model_class = (Qwen2_5_VLForConditionalGeneration
                   if spec['kind'] == 'qwen' else AutoModelForImageTextToText)
    base_model = model_class.from_pretrained(
        spec['base_id'], quantization_config=quantization,
        device_map='auto', dtype=torch.float16,
    ).eval()
    return processor, PeftModel.from_pretrained(base_model, spec['adapter']).eval()

def generate_answer(processor, model, image, prompt, max_new_tokens):
    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'image': image}, {'type': 'text', 'text': prompt},
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], padding=True, return_tensors='pt').to(model.device)
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = output[:, inputs.input_ids.shape[1]:]
    return processor.batch_decode(generated, skip_special_tokens=True)[0].strip()

def evaluate_variant(model_name, spec):
    processor, model = load_variant(spec)
    rows = []
    started = time.perf_counter()
    for i, row in enumerate(gqa_eval, start=1):
        prompt = f"Вопрос: {row['question']}\nОтветь одним словом на русском языке, без пояснений."
        prediction = generate_answer(processor, model, row['image'], prompt, 12)
        rows.append({
            'benchmark': 'GQA-ru', 'sample_id': row['id'], 'question': row['question'],
            'target': row['answer'], 'prediction': prediction,
            'is_correct': normalize_gqa(prediction) == normalize_gqa(row['answer']),
        })
        if i % 10 == 0:
            print(f'{model_name} | GQA-ru: {i}/100')
    for i, row in enumerate(mmbench_eval, start=1):
        options = ' | '.join(f'{letter}. {row[letter]}' for letter in 'ABCD')
        prompt = f"Вопрос: {row['question']}\n{options}\nВыбери правильный вариант. Ответь только буквой A, B, C или D."
        prediction = generate_answer(processor, model, row['image'], prompt, 8)
        selected = extract_option(prediction)
        rows.append({
            'benchmark': 'MMBench-ru', 'sample_id': row['index'], 'question': row['question'],
            'target': row['answer'], 'prediction': prediction, 'selected_option': selected,
            'is_correct': selected == row['answer'],
        })
        if i % 10 == 0:
            print(f'{model_name} | MMBench-ru: {i}/100')
    result = pd.DataFrame(rows)
    result.to_csv(spec['predictions'], index=False)
    print(f'{model_name}: завершено за {(time.perf_counter() - started) / 60:.1f} мин.')
    del model, processor
    gc.collect()
    torch.cuda.empty_cache()
    return result

In [4]:
prediction_frames = []
baseline['model'] = 'B0'
prediction_frames.append(baseline)

for model_name, spec in MODEL_REGISTRY.items():
    if spec['predictions'].exists():
        print(f'{model_name}: использую сохранённые предсказания.')
        result = pd.read_csv(spec['predictions'])
    elif (spec['adapter'] / 'adapter_model.safetensors').exists():
        print(f'{model_name}: начинаю оценку обученного адаптера.')
        result = evaluate_variant(model_name, spec)
    else:
        print(f'{model_name}: готовые веса адаптера не найдены — пропускаю.')
        continue
    result['model'] = model_name
    prediction_frames.append(result)

all_predictions = pd.concat(prediction_frames, ignore_index=True)
all_predictions.to_csv(RESULTS_DIR / 'all_model_predictions.csv', index=False)
display(all_predictions.groupby(['model', 'benchmark']).size().rename('examples'))

F1-fast: использую сохранённые предсказания.
F2: начинаю оценку обученного адаптера.


W0715 02:40:48.724000 20976 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

c:\Users\miste\miniconda3\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


...

A1-InternVL3 | MMBench-ru: 70/100


A1-InternVL3 | MMBench-ru: 80/100


A1-InternVL3 | MMBench-ru: 90/100


A1-InternVL3 | MMBench-ru: 100/100
A1-InternVL3: завершено за 1.5 мин.


model         benchmark 
A1-InternVL3  GQA-ru        100
              MMBench-ru    100
B0            GQA-ru        100
              MMBench-ru    100
F1-fast       GQA-ru        100
              MMBench-ru    100
F2            GQA-ru        100
              MMBench-ru    100
Name: examples, dtype: int64

## 3. Exact Match, Accuracy и BERTScore

BERTScore применяется только к открытым ответам GQA-ru. Для MMBench-ru семантическая близость букв не имеет смысла, поэтому используется Accuracy.

In [5]:
gqa_predictions = all_predictions.query("benchmark == 'GQA-ru'").copy()
precision, recall, f1 = score(
    gqa_predictions['prediction'].fillna('').astype(str).tolist(),
    gqa_predictions['target'].fillna('').astype(str).tolist(),
    model_type=BERTSCORE_MODEL, lang='ru', device=BERTSCORE_DEVICE,
    batch_size=8, rescale_with_baseline=False, verbose=True,
)
gqa_predictions['bertscore_precision'] = precision.cpu().numpy()
gqa_predictions['bertscore_recall'] = recall.cpu().numpy()
gqa_predictions['bertscore_f1'] = f1.cpu().numpy()
gqa_predictions.to_csv(RESULTS_DIR / 'semantic_predictions_gqa.csv', index=False)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/28 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/50 [00:00<?, ?it/s]

done in 3.38 seconds, 118.26 sentences/sec


In [6]:
strict_metrics = (all_predictions.groupby(['model', 'benchmark'])['is_correct']
                  .mean().mul(100).unstack('benchmark'))
strict_metrics = strict_metrics.rename(columns={
    'GQA-ru': 'gqa_exact_match', 'MMBench-ru': 'mmbench_accuracy',
})
semantic_metrics = (gqa_predictions.groupby('model')[
    ['bertscore_precision', 'bertscore_recall', 'bertscore_f1']
].mean().mul(100))
metrics = strict_metrics.join(semantic_metrics).round(2)
metrics['gqa_exact_change_vs_b0'] = (metrics['gqa_exact_match'] - metrics.loc['B0', 'gqa_exact_match']).round(2)
metrics['gqa_semantic_change_vs_b0'] = (metrics['bertscore_f1'] - metrics.loc['B0', 'bertscore_f1']).round(2)
metrics.to_csv(RESULTS_DIR / 'all_model_metrics.csv')
display(metrics.sort_values('bertscore_f1', ascending=False))
print('Сохранены results/all_model_predictions.csv и results/all_model_metrics.csv')

,gqa_exact_match,mmbench_accuracy,bertscore_precision,bertscore_recall,bertscore_f1,gqa_exact_change_vs_b0,gqa_semantic_change_vs_b0
model,,,,,,,
B0,43.0,74.0,87.949997,87.089996,87.470001,0.0,0.00
F2,43.0,74.0,85.430000,84.889999,85.110001,0.0,-2.36
F1-fast,43.0,74.0,85.370003,84.720001,85.000000,0.0,-2.47
A1-InternVL3,27.0,69.0,83.489998,84.260002,83.830002,-16.0,-3.64


Сохранены results/all_model_predictions.csv и results/all_model_metrics.csv


## 4. Выводы по экспериментам

- Исходная Qwen B0 остаётся лучшей по совокупности метрик: 43% GQA Exact Match, 74% MMBench Accuracy и 87,47% BERTScore F1.
- F1-fast сохранила строгие метрики baseline, но снизила семантическую близость до 85,00%. Она изменила 59 из 200 ответов: 7 ошибок исправила и 7 правильных ответов испортила.
- F2 с LoRA rank 16 — лучший из обученных адаптеров: строгие метрики также равны baseline, а BERTScore F1 составляет 85,11%, на 0,11 п.п. выше F1. F2 изменила 71 ответ, исправив 9 ошибок и добавив 9 новых.
- A1 InternVL3 заметно уступила Qwen: 27% GQA, 69% MMBench и 83,83% BERTScore F1. Ограничение одним визуальным патчем и небольшой объём обучения могли снизить качество.
- Главный вывод первой итерации: 2 000 примеров недостаточно для устойчивого улучшения Qwen. Для F3 выбирается Qwen2.5-VL-3B с QLoRA rank 16 и 10 000 примеров — объём, который можно обучить примерно за 8–10 часов.